# Name That LLM: two-stage LoRA classifier on Qwen3-8B-Base (training)

Generated by `name_that_llm/kaggle_lora_train.py`; do not edit by hand.

* **Backbone:** first 24 of 36 layers of Qwen3-8B-Base, LoRA r16 / alpha 32 on all linear layers, final norm off.
* **Head:** mean-pooled hidden states at 8 depths, a LayerNorm each, one linear layer.
* **Stage 1:** 1 epoch on up to 5,000 match-weighted Open Pool passages per model (draw seed 3). **Stage 2:** 2 epochs on train.csv.
* **Check:** stage 2 is run once without prompt-grouped fold 0, to score that fold, and once on all of train.csv for the final checkpoint.

Needs GPU T4 (one is used), internet off, and three inputs: the competition data, `crtproduction/name-that-llm-open-pool` and `qwen-lm/qwen-3/Transformers/8b-base`.

To switch between the timing run and the full run, change `MODE` in the next cell.

In [ ]:
import gc
import glob
import json
import os
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold

MODE = 'full'  # 'timing' or 'full'
T0 = time.time()
ON_KAGGLE = os.path.exists('/kaggle/input')
BASE_SPEC = {'model_type': 'qwen3', 'hidden_size': 4096, 'num_hidden_layers': 36}  # config.json fields of qwen-lm/qwen-3/Transformers/8b-base


def find_base(spec):
    for cfg in sorted(glob.glob('/kaggle/input/**/config.json', recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if all(c.get(k) == v for k, v in spec.items()):
            return os.path.dirname(cfg)
    raise FileNotFoundError(f'attach the base model {spec}')


if ON_KAGGLE:
    attached = sorted(glob.glob('/kaggle/input/*/*'))
    print('attached inputs:', attached)
    comps = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/train.csv', recursive=True)
             if os.path.exists(os.path.join(os.path.dirname(p), 'test.csv'))]
    assert comps, f'Attach the name-that-llm competition data; found {attached}'
    COMP = comps[0]
    pools = glob.glob('/kaggle/input/**/ntl_open_pool.parquet', recursive=True)
    assert pools, f'Attach crtproduction/name-that-llm-open-pool; found {attached}'
    POOL = pools[0]
    BASE = find_base(BASE_SPEC)
    OUT = 'lora_out_8b_big'
    SMOKE = False
else:  # smoke test from this project's notebooks/ folder
    COMP, POOL = '../data/raw', '../data/external/open_pool/ntl_open_pool.parquet'
    BASE = os.environ.get('NTL_SMOKE_MODEL', '../data/interim/tiny_qwen3')
    OUT = '../data/interim/lora_smoke_out'
    SMOKE = True

CFG = dict(
    keep=24, layers=[2, 4, 6, 8, 12, 16, 20, 24], r=16, alpha=32, dropout=0.05,
    max_len=384, bs=8,
    stage1=dict(epochs=1, lr_lora=1e-4, lr_head=1e-3),
    stage2=dict(epochs=2, lr_lora=5e-5, lr_head=5e-4),
    n_per_label=5000, pool_seed=3, holdout_fold=0, folds=5, seed=0,
    base=BASE_SPEC,
)
if SMOKE:
    CFG.update(keep=2, layers=[1, 2], max_len=64, bs=4, n_per_label=10)
N_PER_LABEL, POOL_SEED = CFG['n_per_label'], CFG['pool_seed']

if torch.cuda.is_available():
    DEVICE = torch.device('cuda:0')
    AMP = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
    print(torch.cuda.get_device_name(0), 'amp', AMP)
else:
    DEVICE, AMP = torch.device('cpu'), torch.float32
print('mode', MODE, '| base model', BASE)

## LoRA classifier (copied from `name_that_llm/lora.py`)

In [ ]:
import json
import math
import os
import time

import numpy as np
import torch
from torch import nn
from transformers import AutoConfig, AutoModel, AutoTokenizer

LORA_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")


class LoRALinear(nn.Module):
    """A frozen linear layer plus a trainable low-rank update ``B @ A * alpha / r``."""

    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        dev = base.weight.device
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features, device=dev))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r, device=dev))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        delta = self.drop(x).to(self.lora_A.dtype) @ self.lora_A.T @ self.lora_B.T
        return self.base(x) + (delta * self.scale).to(x.dtype)


def add_lora(model: nn.Module, r: int, alpha: float, dropout: float) -> int:
    """Wrap every target linear layer of ``model`` in LoRALinear; returns how many."""
    targets = [
        (parent, name)
        for parent in model.modules()
        for name, child in parent.named_children()
        if isinstance(child, nn.Linear) and name in LORA_TARGETS
    ]
    for parent, name in targets:
        setattr(parent, name, LoRALinear(getattr(parent, name), r, alpha, dropout))
    return len(targets)


def load_trunk(base_dir: str, keep: int, dtype, device) -> nn.Module:
    """The first ``keep`` decoder layers of the base model, frozen, final norm switched off."""
    cfg = AutoConfig.from_pretrained(base_dir)
    cfg.num_hidden_layers, cfg.use_cache = keep, False
    if getattr(cfg, "layer_types", None) is not None:
        cfg.layer_types = cfg.layer_types[:keep]
    trunk = AutoModel.from_pretrained(base_dir, config=cfg, dtype=dtype).to(device)
    trunk.norm = nn.Identity()
    trunk.requires_grad_(False)
    return trunk


class LoRAClassifier(nn.Module):
    def __init__(
        self, trunk: nn.Module, layers, n_classes: int, r=16, alpha=32, dropout=0.05, ckpt=True
    ):
        super().__init__()
        self.trunk, self.layers = trunk, list(layers)
        self.n_lora = add_lora(trunk, r, alpha, dropout)
        if ckpt:  # recompute activations in backward: far less memory for ~30% more time
            trunk.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs={"use_reentrant": False}
            )
        H = trunk.config.hidden_size
        dev = trunk.embed_tokens.weight.device
        self.norms = nn.ModuleList([nn.LayerNorm(H) for _ in self.layers]).to(dev)
        self.head = nn.Linear(H * len(self.layers), n_classes).to(dev)

    def forward(self, input_ids, attention_mask):
        hs = self.trunk(
            input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True
        ).hidden_states
        m = attention_mask.unsqueeze(-1).float()
        m[:, 0] = 0  # the first token is an attention sink; leave it out of the mean
        den = m.sum(1).clamp(min=1)
        with torch.autocast(device_type=input_ids.device.type, enabled=False):
            z = torch.cat(
                [nm((hs[l].float() * m).sum(1) / den) for l, nm in zip(self.layers, self.norms)],
                -1,
            )
            return self.head(z)

    def trainable_state(self) -> dict:
        """Only what training changes: LoRA matrices, LayerNorms and the head."""
        return {k: v.detach().cpu() for k, v in self.named_parameters() if v.requires_grad}


def tokenize(tok, texts, max_len: int) -> list[list[int]]:
    ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=max_len - 1)
    return [x + [tok.eos_token_id] for x in ids["input_ids"]]


def batches(ids, bs: int, shuffle: bool, rng=None):
    """Index batches of similar length (less padding); batch order shuffled when training."""
    if not shuffle:
        order = np.argsort([len(x) for x in ids], kind="stable")
    else:  # sort within random chunks of 50 batches so batches still differ between epochs
        order = rng.permutation(len(ids))
        chunk = bs * 50
        order = np.concatenate(
            [
                sorted(order[i : i + chunk], key=lambda j: len(ids[j]))
                for i in range(0, len(order), chunk)
            ]
        )
    out = [order[i : i + bs] for i in range(0, len(order), bs)]
    if shuffle:
        out = [out[i] for i in rng.permutation(len(out))]
    return out


def collate(tok, ids, idx, device):
    enc = tok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt")
    return enc["input_ids"].to(device), enc["attention_mask"].to(device)


def train_stage(
    model,
    tok,
    texts,
    y,
    *,
    epochs: float,
    lr_lora: float,
    lr_head: float,
    bs: int,
    max_len: int,
    n_classes: int,
    amp_dtype,
    seed: int = 0,
    max_steps: int | None = None,
    log_every: int = 50,
    log=print,
):
    """One training stage; class-weighted CE, AdamW, 10% warm-up then cosine decay."""
    device = model.head.weight.device
    rng = np.random.default_rng(seed)
    ids = tokenize(tok, texts, max_len)
    y = torch.tensor(np.array(y), device=device)
    counts = torch.bincount(y, minlength=n_classes).float()
    class_w = (counts.sum() / (n_classes * counts.clamp(min=1))).to(device)

    per_epoch = math.ceil(len(ids) / bs)
    total = math.ceil(epochs * per_epoch)
    if max_steps is not None:
        total = min(total, max_steps)
    lora = [p for n, p in model.named_parameters() if p.requires_grad and "lora_" in n]
    rest = [p for n, p in model.named_parameters() if p.requires_grad and "lora_" not in n]
    opt = torch.optim.AdamW(
        [{"params": lora, "lr": lr_lora}, {"params": rest, "lr": lr_head}], weight_decay=0.01
    )
    warm = max(1, int(0.1 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt,
        lambda s: (
            (s + 1) / warm
            if s < warm
            else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))
        ),
    )
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and amp_dtype == torch.float16)

    model.train()
    step, skipped, run_loss, run_n, t0 = 0, 0, 0.0, 0, time.time()
    while step < total:
        for idx in batches(ids, bs, shuffle=True, rng=rng):
            if step >= total:
                break
            input_ids, mask = collate(tok, ids, idx, device)
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                logits = model(input_ids, mask)
            loss = nn.functional.cross_entropy(logits.float(), y[idx], weight=class_w)
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                skipped += 1  # fp16 overflow in the backbone; drop this batch
            else:
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(lora + rest, 1.0)
                scaler.step(opt)
                scaler.update()
                run_loss += loss.item()
                run_n += 1
            sched.step()
            step += 1
            if step % log_every == 0 or step == total:
                el = time.time() - t0
                log(
                    f"step {step}/{total} loss {run_loss / max(1, run_n):.4f} "
                    f"{el / step:.2f}s/step, ~{el / step * (total - step) / 60:.0f} min left, "
                    f"skipped {skipped}"
                )
                run_loss, run_n = 0.0, 0
    model.eval()
    return {"steps": total, "skipped": skipped, "seconds": time.time() - t0}


@torch.no_grad()
def predict_logits(model, tok, texts, *, bs: int, max_len: int, amp_dtype) -> np.ndarray:
    device = model.head.weight.device
    ids = tokenize(tok, texts, max_len)
    out = np.zeros((len(ids), model.head.out_features), np.float32)
    model.eval()
    for idx in batches(ids, bs, shuffle=False):
        input_ids, mask = collate(tok, ids, idx, device)
        with torch.autocast(
            device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda"
        ):
            out[idx] = model(input_ids, mask).float().cpu().numpy()
    return out


def load_tokenizer(base_dir: str):
    tok = AutoTokenizer.from_pretrained(base_dir)
    tok.padding_side = "right"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


def save_checkpoint(model, out_dir: str, meta: dict):
    from safetensors.torch import save_file

    os.makedirs(out_dir, exist_ok=True)
    save_file(
        {k: v.contiguous() for k, v in model.trainable_state().items()},
        os.path.join(out_dir, "lora_classifier.safetensors"),
    )
    with open(os.path.join(out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)


def load_checkpoint(model, ckpt_dir: str):
    from safetensors.torch import load_file

    state = load_file(os.path.join(ckpt_dir, "lora_classifier.safetensors"))
    missing = model.load_state_dict(state, strict=False)
    assert not missing.unexpected_keys, missing.unexpected_keys
    return model


## Data

In [ ]:
train = pd.read_csv(f'{COMP}/train.csv')
pool = pd.read_parquet(POOL, columns=['text', 'model', 'match_weight'])

# same draw as name_that_llm/dataset.py: weighted sampling without replacement
# (Efraimidis-Spirakis) by match_weight, the same number of passages per label
parts = []
for label, d in pool.groupby('model'):
    rng = np.random.default_rng([POOL_SEED, len(parts)])
    keys = rng.random(len(d)) ** (1.0 / d.match_weight.clip(lower=1e-6).to_numpy())
    parts.append(d.iloc[np.argsort(-keys)[:min(N_PER_LABEL, len(d))]])
pool_sample = pd.concat(parts, ignore_index=True)
del pool

if SMOKE:
    train = train.groupby('model').head(12).reset_index(drop=True)
LABELS = sorted(train.model.unique())
L2I = {m: i for i, m in enumerate(LABELS)}
assert len(LABELS) == 18 and set(pool_sample.model) <= set(LABELS)
y_train = train.model.map(L2I).to_numpy()
y_pool = pool_sample.model.map(L2I).to_numpy()

folds = list(GroupKFold(n_splits=CFG['folds']).split(train, y_train, train.prompt_id))
fit_idx, hold_idx = folds[CFG['holdout_fold']]
print(f'train {len(train)} (holdout {len(hold_idx)}), pool sample {len(pool_sample)}')

## Model

In [ ]:
tok = load_tokenizer(BASE)
trunk = load_trunk(BASE, CFG['keep'], AMP if DEVICE.type == 'cuda' else torch.float32, DEVICE)
model = LoRAClassifier(trunk, CFG['layers'], len(LABELS), CFG['r'], CFG['alpha'], CFG['dropout'])
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'{model.n_lora} LoRA layers, {n_train / 1e6:.1f}M trainable parameters, '
      f'loaded in {time.time() - T0:.0f}s')

common = dict(bs=CFG['bs'], max_len=CFG['max_len'], n_classes=len(LABELS), amp_dtype=AMP)

## Stage 1: the Open Pool sample

In `timing` mode this runs only 60 steps and forecasts a full run.

In [ ]:
if DEVICE.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()
timing = MODE == 'timing'
s1 = train_stage(model, tok, pool_sample.text, y_pool, **CFG['stage1'], **common,
                 seed=CFG['seed'], max_steps=60 if timing else None, log_every=20 if timing else 200)
print('stage 1:', s1)
if DEVICE.type == 'cuda':
    print(f'peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB')
if timing:
    per_step = s1['seconds'] / s1['steps']
    bs = CFG['bs']
    steps = {
        'stage 1': -(-len(pool_sample) // bs),
        'stage 2 without holdout': 2 * -(-len(fit_idx) // bs),
        'stage 2 on all': 2 * -(-len(train) // bs),
    }
    for k, v in steps.items():
        print(f'{k}: {v} steps, ~{v * per_step / 60:.0f} min')
    print(f'forecast for MODE=full: ~{sum(steps.values()) * per_step / 3600:.1f} h '
          f'of training plus loading and prediction')
else:
    stage1_state = {k: v.clone() for k, v in model.trainable_state().items()}

## Stage 2 without the held-out fold, then score that fold

In [ ]:
if MODE == 'full':
    s2 = train_stage(model, tok, train.text.iloc[fit_idx], y_train[fit_idx], **CFG['stage2'], **common,
                     seed=CFG['seed'] + 1, log_every=100)
    logits = predict_logits(model, tok, train.text.iloc[hold_idx], bs=2 * CFG['bs'],
                            max_len=CFG['max_len'], amp_dtype=AMP)
    pred = logits.argmax(1)
    hold_bacc = balanced_accuracy_score(y_train[hold_idx], pred)
    print(f'stage 2: {s2}\nheld-out fold: balanced accuracy {hold_bacc:.4f} '
          f'({len(hold_idx)} passages, {train.prompt_id.iloc[hold_idx].nunique()} prompts)')
    os.makedirs(OUT, exist_ok=True)
    hold = pd.DataFrame(logits, columns=LABELS)
    hold.insert(0, 'id', train.id.iloc[hold_idx].to_numpy())
    hold.insert(1, 'model', train.model.iloc[hold_idx].to_numpy())
    hold.to_parquet(f'{OUT}/holdout_logits.parquet', index=False)

## Stage 2 on all of train.csv, from the same stage-1 weights; save

In [ ]:
if MODE == 'full':
    model.load_state_dict({k: v.to(DEVICE) for k, v in stage1_state.items()}, strict=False)
    s2_all = train_stage(model, tok, train.text, y_train, **CFG['stage2'], **common,
                         seed=CFG['seed'] + 2, log_every=100)
    save_checkpoint(model, OUT, dict(labels=LABELS, cfg=CFG, holdout_balanced_accuracy=hold_bacc,
                                     stage1=s1, stage2_holdout=s2, stage2_all=s2_all))
    print('saved', sorted(os.listdir(OUT)))
print(f'done in {(time.time() - T0) / 60:.1f} min')